In [2]:
import pandas as pd
import numpy as np
import re

In [3]:
calendario = pd.read_csv(r"C:\Users\bario\OneDrive\Documentos\UC3M\PROYECTO EN CIENCIA DE DATOS\proyecto_emergencias-20260915T155207Z-1-001\proyecto_emergencias\calendario.csv")
calendario.head()

,fecha,dia_semana,dia_semana_num,semana_año,mes,trimestre,tipo_dia,festivo,evento,tipo_evento,zona_evento,estacion,es_vispera_festivo,vacaciones_escolares
0,2024-01-01,Monday,0,1,1,1,festivo,Año Nuevo,NaN,NaN,NaN,invierno,0,1
1,2024-01-02,Tuesday,1,1,1,1,laborable,NaN,NaN,NaN,NaN,invierno,0,1
2,2024-01-03,Wednesday,2,1,1,1,laborable,NaN,NaN,NaN,NaN,invierno,0,1
3,2024-01-04,Thursday,3,1,1,1,laborable,NaN,NaN,NaN,NaN,Invierno,0,1
4,2024-01-05,Friday,4,1,1,1,puente,NaN,NaN,NaN,NaN,invierno,1,1


In [ ]:
calendario_limpio = calendario.copy()

In [15]:
calendario_limpio.head()

,fecha,dia_semana,dia_semana_num,semana_año,mes,trimestre,tipo_dia,festivo,evento,tipo_evento,zona_evento,estacion,es_vispera_festivo,vacaciones_escolares
0,01-01-2024,LUNES,0,1,1,1,festivo,Año Nuevo,NaN,NaN,NaN,invierno,0,1
1,02-01-2024,MARTES,1,1,1,1,laborable,NaN,NaN,NaN,NaN,invierno,0,1
2,03-01-2024,MIERCOLES,2,1,1,1,laborable,NaN,NaN,NaN,NaN,invierno,0,1
3,04-01-2024,JUEVES,3,1,1,1,laborable,NaN,NaN,NaN,NaN,Invierno,0,1
4,05-01-2024,VIERNES,4,1,1,1,puente,NaN,NaN,NaN,NaN,invierno,1,1


**EMPEZAMOS CON UNA ANALISIS DEL DATASET PARA VER LAS INCONSISTENCIAS EXISTENTES**
---

In [22]:
# Conteo y visualización de filas 100% idénticas
filas_duplicadas_exactas = calendario_limpio[calendario_limpio.duplicated(keep=False)]

print("===== CONSULTA DE DUPLICADOS EXACTOS =====")
print(f"• Total de filas idénticas encontradas: {calendario_limpio.duplicated().sum()}")
print(f"• Total de registros involucrados en las parejas: {len(filas_duplicadas_exactas)}")

# Mostrar la tabla con los registros idénticos
display(filas_duplicadas_exactas.sort_values('fecha'))

===== CONSULTA DE DUPLICADOS EXACTOS =====
• Total de filas idénticas encontradas: 0
• Total de registros involucrados en las parejas: 0


,fecha,dia_semana,dia_semana_num,semana_año,mes,trimestre,tipo_dia,festivo,evento,tipo_evento,zona_evento,estacion,es_vispera_festivo,vacaciones_escolares


Hay 5 filas completamente iguales, eliminar sus duplicados: 

In [6]:
# Eliminar filas duplicadas exactas conservando la primera aparición
calendario_limpio = calendario_limpio.drop_duplicates().reset_index(drop=True)

# Comprobación de que el total de filas coincide con los 366 días de 2024
print("===== RESULTADO TRAS ELIMINAR DUPLICADOS =====")
print(f"• Total de filas actuales: {len(calendario_limpio)} (Esperado: 366)")
print(f"• Duplicados exactos restantes: {calendario_limpio.duplicated().sum()}")

===== RESULTADO TRAS ELIMINAR DUPLICADOS =====
• Total de filas actuales: 366 (Esperado: 366)
• Duplicados exactos restantes: 0


In [29]:

print("=" * 70)
print("ANALISIS INICIAL DE FORMATOS Y VALORES EN 'calendario_limpio'")
print("=" * 70)

# 1. ANÁLISIS GENERAL
total_filas = len(calendario_limpio)
print(f"\n1. ANÁLISIS GENERAL")
print(f"• Total de filas registradas: {total_filas}")
print(f"• Total de columnas: {len(calendario_limpio.columns)}")

# 2. ANÁLISIS DETALLADO DE LA COLUMNA 'fecha'
print(f"\n 2. ANÁLISIS COLUMNA 'fecha'")
nulos_fecha = calendario_limpio['fecha'].isnull().sum()
print(f"• Total de valores nulos en 'fecha': {nulos_fecha}")

# Función para identificar el separador utilizado
def detectar_separador(val):
    if pd.isnull(val): return "Nulo"
    val_str = str(val)
    if '-' in val_str: return "Guión (-)"
    elif '/' in val_str: return "Barra (/)"
    elif '.' in val_str: return "Punto (.)"
    else: return "Otro / Sin separador"

# Función para identificar el orden de la estructura (YYYY-MM-DD vs DD-MM-YYYY)
def detectar_estructura(val):
    if pd.isnull(val): return "Nulo"
    partes = re.split(r'[-/.]', str(val))
    if len(partes) == 3:
        p1, p2, p3 = partes
        if len(p1) == 4:
            return "Año - Mes - Día"
        elif len(p3) == 4:
            return "Día - Mes - Año"
        elif len(p1) <= 2 and len(p3) <= 2:
            return "Día/Mes/Año"
    return "Estructura no identificada"

# Aplicar análisis
separadores = calendario_limpio['fecha'].apply(detectar_separador).value_counts()
estructuras = calendario_limpio['fecha'].apply(detectar_estructura).value_counts()

print("• Conteo por tipo de SEPARADOR en fecha:")
for sep, cant in separadores.items():
    print(f"  - {sep}: {cant} registros ({round(cant/total_filas*100, 2)}%)")

print("• Conteo por ESTRUCTURA / ORDEN de la fecha:")
for est, cant in estructuras.items():
    print(f"  - {est}: {cant} registros ({round(cant/total_filas*100, 2)}%)")

# 3. ANÁLISIS DE 'dia_semana' Y 'dia_semana_num'
print(f"\n 3. ANÁLISIS DE 'dia_semana' Y SU MAPEO NUMÉRICO")
print("• Variantes de escritura encontradas en 'dia_semana':")
variantes_dias = calendario_limpio['dia_semana'].value_counts(dropna=False)
for dia, cant in variantes_dias.items():
    print(f"  - '{dia}': {cant} apariciones")

print("\n• Tabla de correspondencia entre 'dia_semana', 'dia_semana_num' y su frecuencia:")
mapeo_dias = calendario_limpio.groupby(['dia_semana', 'dia_semana_num'], dropna=False).size().reset_index(name='frecuencia')
display(mapeo_dias)

# 4. VARIABLES TEMPORALES (semana_año, mes, trimestre)
print(f"\n 4. VALORES Y FRECUENCIAS EN VARIABLES TEMPORALES")

cols_temporales = ['semana_año', 'mes', 'trimestre']
for col in cols_temporales:
    if col in calendario_limpio.columns:
        print(f"\n--- Columna '{col}' ---")
        conteo = calendario_limpio[col].value_counts(dropna=False).sort_index()
        for val, freq in conteo.items():
            print(f"  - El valor [{int(val) if pd.notnull(val) else 'Nulo'}] se repite {freq} veces")

# 5. VALORES ÚNICOS Y VARIACIONES EN COLUMNAS CATEGÓRICAS
print(f"\n 5. VALORES ÚNICOS Y VARIACIONES EN COLUMNAS CATEGÓRICAS")

cols_categoricas = ['tipo_dia', 'festivo', 'evento', 'tipo_evento', 'zona_evento', 
                    'estacion', 'es_vispera_festivo', 'vacaciones_escolares']

for col in cols_categoricas:
    if col in calendario_limpio.columns:
        unite = calendario_limpio[col].unique()
        nulos_col = calendario_limpio[col].isnull().sum()
        print(f"\n--- Columna: '{col}' (Nulos: {nulos_col} | {round(nulos_col/total_filas*100, 2)}%) ---")
        print(f"Valores únicos ({len(unite)} distintos): {list(unite[:15])}")

ANALISIS INICIAL DE FORMATOS Y VALORES EN 'calendario_limpio'

1. ANÁLISIS GENERAL
• Total de filas registradas: 366
• Total de columnas: 14

 2. ANÁLISIS COLUMNA 'fecha'
• Total de valores nulos en 'fecha': 0
• Conteo por tipo de SEPARADOR en fecha:
  - Guión (-): 366 registros (100.0%)
• Conteo por ESTRUCTURA / ORDEN de la fecha:
  - Día - Mes - Año: 366 registros (100.0%)

 3. ANÁLISIS DE 'dia_semana' Y SU MAPEO NUMÉRICO
• Variantes de escritura encontradas en 'dia_semana':
  - 'LUNES': 53 apariciones
  - 'MARTES': 53 apariciones
  - 'MIERCOLES': 52 apariciones
  - 'JUEVES': 52 apariciones
  - 'VIERNES': 52 apariciones
  - 'SABADO': 52 apariciones
  - 'DOMINGO': 52 apariciones

• Tabla de correspondencia entre 'dia_semana', 'dia_semana_num' y su frecuencia:


,dia_semana,dia_semana_num,frecuencia
0,DOMINGO,6,52
1,JUEVES,3,52
2,LUNES,0,53
3,MARTES,1,53
4,MIERCOLES,2,52
5,SABADO,5,52
6,VIERNES,4,52



 4. VALORES Y FRECUENCIAS EN VARIABLES TEMPORALES

--- Columna 'semana_año' ---
  - El valor [1] se repite 7 veces
  - El valor [2] se repite 7 veces
  - El valor [3] se repite 7 veces
  - El valor [4] se repite 7 veces
  - El valor [5] se repite 7 veces
  - El valor [6] se repite 7 veces
  - El valor [7] se repite 7 veces
  - El valor [8] se repite 7 veces
  - El valor [9] se repite 7 veces
  - El valor [10] se repite 7 veces
  - El valor [11] se repite 7 veces
  - El valor [12] se repite 7 veces
  - El valor [13] se repite 7 veces
  - El valor [14] se repite 7 veces
  - El valor [15] se repite 7 veces
  - El valor [16] se repite 7 veces
  - El valor [17] se repite 7 veces
  - El valor [18] se repite 7 veces
  - El valor [19] se repite 7 veces
  - El valor [20] se repite 7 veces
  - El valor [21] se repite 7 veces
  - El valor [22] se repite 7 veces
  - El valor [23] se repite 7 veces
  - El valor [24] se repite 7 veces
  - El valor [25] se repite 7 veces
  - El valor [26] se repite 

- Festivo y evento juntar columna?
- zona evento --> distrito
- es vispera y vacaciones son 0 y 1, lo querenmos tratar asi o con True y False?? 1 es si o es no?


### OBSERVACIONES 

1. Fechas Inconsistentes: No hay nulos pero se mezclan separadores con 344 registros en guión y 27 en barra. Además hay distintas estructuras con 347 registros en AAAA-MM-DD y 24 en DD-MM-AAAA. Hay un exceso de filas con 371 registros para un año de 366 días lo que indica posibles duplicados. Solución -> homogeneizar todo al formato estricto DD-MM-YYYY y auditar duplicados.
2. Fragmentación Categórica en Día Semana, Estación y Tipo Día: El campo dia_semana presenta 39 variantes sintácticas para solo 7 días por mezcla de idiomas, mayúsculas, minúsculas, abreviaturas y códigos de una letra. El campo estacion presenta 17 variantes para 4 estaciones por inglés, español y mayúsculas. El campo tipo_dia presenta 16 variantes como festivo, laborable, puente, finde, FS, F, fiesta o variantes en mayúsculas. Solución -> mapear con diccionarios y transformar a formato estandarizado.
3. Alta Densidad de Nulos en Eventos y Festivos: Elevado porcentaje de nulos estructurales con festivo al 96.5%, evento al 95.96%, tipo_evento al 95.96% y zona_evento al 96.77%. Solución -> rellenar los nulos con etiquetas explícitas como Sin Festivo o Sin Evento.
4. Semana 1 con 9 días. Solución --> Añadir semana 53 para 30 y 31 de diciembre.

**SOLUCÍON FORMATO FECHA**

In [7]:
#Función flexible para detectar el origen de la fecha
def parsear_fecha_flexible(fecha_str):
    if pd.isnull(fecha_str):
        return pd.NaT
    fecha_str = str(fecha_str).strip()
    
    # Si la cadena empieza por año 
    if fecha_str.startswith('2024'):
        return pd.to_datetime(fecha_str, format='mixed', dayfirst=False)
    # Si empieza por día 
    else:
        return pd.to_datetime(fecha_str, format='mixed', dayfirst=True)

# Aplicar el parseo flexible
calendario_limpio['fecha_clean'] = calendario_limpio['fecha'].apply(parsear_fecha_flexible)

# Formatear la fecha al estándar exacto DD-MM-YYYY
calendario_limpio['fecha'] = calendario_limpio['fecha_clean'].dt.strftime('%d-%m-%Y')

# Ordenar cronológicamente y descartar la auxiliar
calendario_limpio = calendario_limpio.sort_values('fecha_clean').reset_index(drop=True)
calendario_limpio = calendario_limpio.drop(columns=['fecha_clean'])

# VERIFICACIÓN
print("===== RESULTADOS TRAS EDITAR FECHAS =====")
print(f"- Fechas mal formateadas restantes: {calendario_limpio['fecha'].isnull().sum()}")
print(f"- Separadores '/' restantes: {calendario_limpio['fecha'].str.contains('/').sum()}")
print(f"- Fechas con formato con guion '-': {calendario_limpio['fecha'].str.contains('-').sum()}")
print(f"- Fechas/Días duplicados en el dataset: {calendario_limpio['fecha'].duplicated().sum()}")

# Validar la secuencia de los 366 días de 2024
fechas_dt = pd.to_datetime(calendario_limpio['fecha'], format='%d-%m-%Y')
rango_esperado = pd.date_range(start='2024-01-01', end='2024-12-31', freq='D')
dias_faltantes = len(rango_esperado) - len(fechas_dt.unique())

print(f"- Días faltantes en el año 2024 (esperados 366): {dias_faltantes}")


===== RESULTADOS TRAS EDITAR FECHAS =====
- Fechas mal formateadas restantes: 0
- Separadores '/' restantes: 0
- Fechas con formato con guion '-': 366
- Fechas/Días duplicados en el dataset: 0
- Días faltantes en el año 2024 (esperados 366): 0


**SOLUCIÓN FORMATO DIA DE LA SEMANA**

In [8]:
import pandas as pd

# Diccionario de mapeo exhaustivo sin tildes
mapeo_dias_espanol = {
    # Lunes
    'Monday': 'LUNES', 'LUNES': 'LUNES', 'Lunes': 'LUNES', 'lunes': 'LUNES', 'L': 'LUNES',
    # Martes
    'Tuesday': 'MARTES', 'MARTES': 'MARTES', 'Martes': 'MARTES', 'martes': 'MARTES', 'Tue': 'MARTES', 'M': 'MARTES',
    # Miércoles
    'Wednesday': 'MIERCOLES', 'MIERCOLES': 'MIERCOLES', 'Miércoles': 'MIERCOLES', 'miércoles': 'MIERCOLES', 'Wed': 'MIERCOLES', 'X': 'MIERCOLES',
    # Jueves
    'Thursday': 'JUEVES', 'JUEVES': 'JUEVES', 'Jueves': 'JUEVES', 'jueves': 'JUEVES', 'Thu': 'JUEVES', 'J': 'JUEVES',
    # Viernes
    'Friday': 'VIERNES', 'VIERNES': 'VIERNES', 'Viernes': 'VIERNES', 'viernes': 'VIERNES', 'Fri': 'VIERNES', 'V': 'VIERNES',
    # Sábado
    'Saturday': 'SABADO', 'SABADO': 'SABADO', 'Sábado': 'SABADO', 'sábado': 'SABADO', 'Sat': 'SABADO', 'S': 'SABADO',
    # Domingo
    'Sunday': 'DOMINGO', 'Domingo': 'DOMINGO', 'domingo': 'DOMINGO', 'Sun': 'DOMINGO', 'D': 'DOMINGO'
}

# Aplicar el mapeo sobre la columna dia_semana
calendario_limpio['dia_semana'] = calendario_limpio['dia_semana'].map(mapeo_dias_espanol)

# COMPROBACIÓN Y VERIFICACIÓN
print("===== RESULTADOS TRAS HOMOGENEIZAR DIA_SEMANA (SIN TILDES) =====")
print("• Valores únicos restantes en 'dia_semana':")
print(calendario_limpio['dia_semana'].unique())

print("\n• Conteo por cada día de la semana:")
print(calendario_limpio['dia_semana'].value_counts())

print("\n• Tabla de correspondencia final entre 'dia_semana' y 'dia_semana_num':")
mapeo_final = calendario_limpio.groupby(['dia_semana', 'dia_semana_num']).size().reset_index(name='frecuencia')
display(mapeo_final)

===== RESULTADOS TRAS HOMOGENEIZAR DIA_SEMANA (SIN TILDES) =====
• Valores únicos restantes en 'dia_semana':
<StringArray>
['LUNES', 'MARTES', 'MIERCOLES', 'JUEVES', 'VIERNES', 'SABADO', 'DOMINGO']
Length: 7, dtype: str

• Conteo por cada día de la semana:
dia_semana
LUNES        53
MARTES       53
MIERCOLES    52
JUEVES       52
VIERNES      52
SABADO       52
DOMINGO      52
Name: count, dtype: int64

• Tabla de correspondencia final entre 'dia_semana' y 'dia_semana_num':


,dia_semana,dia_semana_num,frecuencia
0,DOMINGO,6,52
1,JUEVES,3,52
2,LUNES,0,53
3,MARTES,1,53
4,MIERCOLES,2,52
5,SABADO,5,52
6,VIERNES,4,52


**SOLUCIÓN FORMATO TIPO_DIA**

In [27]:
# 1. Limpieza básica de texto
calendario_limpio['tipo_dia'] = calendario_limpio['tipo_dia'].astype(str).str.strip().str.upper()

# 2. Homogeneizar sinónimos a las 4 categorías estándar
reemplazos = {
    'FESTIVO': 'FESTIVO', 'FESTIVA': 'FESTIVO', 'FIESTA': 'FESTIVO', 'F': 'FESTIVO',
    'LABORABLE': 'LABORABLE', 'LABORAL': 'LABORABLE',
    'FIN_SEMANA': 'FIN_DE_SEMANA', 'FIN DE SEMANA': 'FIN_DE_SEMANA', 'FINDE': 'FIN_DE_SEMANA', 'FS': 'FIN_DE_SEMANA',
    'PUENTE': 'PUENTE'
}

calendario_limpio['tipo_dia'] = calendario_limpio['tipo_dia'].replace(reemplazos)

# VERIFICACIÓN DEFINITIVA
print("===== RESULTADOS TRAS RECUPERAR PUENTES =====")
print("• Categorías únicas resultantes:")
print(calendario_limpio['tipo_dia'].unique())

print("\n• Conteo exacto por tipo de día:")
print(calendario_limpio['tipo_dia'].value_counts())

===== RESULTADOS TRAS RECUPERAR PUENTES =====
• Categorías únicas resultantes:
<StringArray>
['FESTIVO', 'LABORABLE', 'PUENTE', 'FIN_DE_SEMANA', 'L']
Length: 5, dtype: str

• Conteo exacto por tipo de día:
tipo_dia
LABORABLE        225
FIN_DE_SEMANA    105
FESTIVO           29
PUENTE             4
L                  3
Name: count, dtype: int64


In [28]:
# Reemplazar 'L' por 'LABORABLE'
calendario_limpio['tipo_dia'] = calendario_limpio['tipo_dia'].replace({'L': 'LABORABLE'})

# VERIFICACIÓN DEFINITIVA
print("===== RESULTADOS FINALES DE TIPO_DIA =====")
print("• Categorías únicas resultantes:")
print(calendario_limpio['tipo_dia'].unique())

print("\n• Conteo exacto por tipo de día (Suma: 366 días):")
print(calendario_limpio['tipo_dia'].value_counts())

===== RESULTADOS FINALES DE TIPO_DIA =====
• Categorías únicas resultantes:
<StringArray>
['FESTIVO', 'LABORABLE', 'PUENTE', 'FIN_DE_SEMANA']
Length: 4, dtype: str

• Conteo exacto por tipo de día (Suma: 366 días):
tipo_dia
LABORABLE        228
FIN_DE_SEMANA    105
FESTIVO           29
PUENTE             4
Name: count, dtype: int64


**SOLUCIÓN FORMATO TIPO_EVENTO**
- Más del 90% no tiene etiqueta, es decir, es Nan, así pues para aquellos valores añadiremos la etiqueta SIN_EVENTO

In [17]:
# 1. Rllenar nulos con la etiqueta explícita 'SIN_EVENTO'
calendario_limpio['tipo_evento'] = calendario_limpio['tipo_evento'].fillna('SIN_EVENTO')

# 2. Diccionario de mapeo para asegurar formato consistente (mayúsculas, sin tildes)
mapeo_tipo_evento = {
    'DEPORTIVO': 'DEPORTIVO',
    'CELEBRACION': 'CELEBRACION',
    'FIESTAS': 'FIESTAS',
    'CONCIERTO': 'CONCIERTO',
    'SIN_EVENTO': 'SIN_EVENTO'
}

calendario_limpio['tipo_evento'] = calendario_limpio['tipo_evento'].map(mapeo_tipo_evento)

# VERIFICACIÓN
print("===== RESULTADOS TRAS HOMOGENEIZAR TIPO_EVENTO =====")
print("• Nulos restantes en 'tipo_evento':", calendario_limpio['tipo_evento'].isna().sum())
print("\n• Categorías únicas resultantes:")
print(calendario_limpio['tipo_evento'].unique())

print("\n• Conteo de frecuencias por tipo de evento:")
print(calendario_limpio['tipo_evento'].value_counts())

===== RESULTADOS TRAS HOMOGENEIZAR TIPO_EVENTO =====
• Nulos restantes en 'tipo_evento': 0

• Categorías únicas resultantes:
<StringArray>
['SIN_EVENTO', 'DEPORTIVO', 'CELEBRACION', 'FIESTAS', 'CONCIERTO']
Length: 5, dtype: str

• Conteo de frecuencias por tipo de evento:
tipo_evento
SIN_EVENTO     352
DEPORTIVO        5
CELEBRACION      4
CONCIERTO        3
FIESTAS          2
Name: count, dtype: int64


**SOLUCIÓN FORMATO ESTACIÓN**

In [18]:
# Diccionario de mapeo completo para todas las variantes encontradas en 'estacion'
mapeo_estacion = {
    # Invierno
    'invierno': 'INVIERNO', 'Invierno': 'INVIERNO', 'INVIERNO': 'INVIERNO',
    # Primavera
    'primavera': 'PRIMAVERA', 'Primavera': 'PRIMAVERA', 'PRIMAVERA': 'PRIMAVERA',
    # Verano
    'verano': 'VERANO', 'Verano': 'VERANO', 'VERANO': 'VERANO', 'Summer': 'VERANO',
    # Otoño
    'otoño': 'OTONO', 'Otoño': 'OTONO', 'OTOÑO': 'OTONO', 'Autumn': 'OTONO', 'Fall': 'OTONO'
}

# Aplicar el mapeo sobre la columna estacion
calendario_limpio['estacion'] = calendario_limpio['estacion'].map(mapeo_estacion)

# VERIFICACIÓN
print("===== RESULTADOS TRAS HOMOGENEIZAR ESTACION =====")
print("• Nulos restantes:", calendario_limpio['estacion'].isna().sum())
print("• Categorías únicas resultantes:")
print(calendario_limpio['estacion'].unique())

print("\n• Conteo de días por estación:")
print(calendario_limpio['estacion'].value_counts())

===== RESULTADOS TRAS HOMOGENEIZAR ESTACION =====
• Nulos restantes: 4
• Categorías únicas resultantes:
<StringArray>
['INVIERNO', 'OTONO', 'VERANO', 'PRIMAVERA', nan]
Length: 5, dtype: str

• Conteo de días por estación:
estacion
OTONO        95
PRIMAVERA    94
VERANO       88
INVIERNO     85
Name: count, dtype: int64


Nos equivocaos y se pusieron 4 celdas como nulos en estacion, sin embargo no habia ninguna celda vacia en esta columna asi que asignaremos la estación según la fecha.

In [20]:
# Función para asignar la estación exacta en función del día y mes
def obtener_estacion(fecha_str):
    # Convertir la fecha string (DD-MM-YYYY) a objeto datetime
    dt = pd.to_datetime(fecha_str, format='%d-%m-%Y')
    mes, dia = dt.month, dt.day
    
    # Definición meteorológica / astronómica estándar
    if (mes == 12 and dia >= 21) or (mes in [1, 2]) or (mes == 3 and dia < 21):
        return 'INVIERNO'
    elif (mes == 3 and dia >= 21) or (mes in [4, 5]) or (mes == 6 and dia < 21):
        return 'PRIMAVERA'
    elif (mes == 6 and dia >= 21) or (mes in [7, 8]) or (mes == 9 and dia < 23):
        return 'VERANO'
    else:
        return 'OTONO'

# Asignar la estación limpia en base a la columna fecha
calendario_limpio['estacion'] = calendario_limpio['fecha'].apply(obtener_estacion)

# VERIFICACIÓN COMPLETA
print("===== VERIFICACIÓN DEFINITIVA DE ESTACIÓN =====")
print("• Nulos restantes en 'estacion':", calendario_limpio['estacion'].isna().sum())
print("• Categorías únicas resultantes:")
print(calendario_limpio['estacion'].unique())

print("\n• Conteo de días por estación (Total = 366 días):")
print(calendario_limpio['estacion'].value_counts())

===== VERIFICACIÓN DEFINITIVA DE ESTACIÓN =====
• Nulos restantes en 'estacion': 0
• Categorías únicas resultantes:
<StringArray>
['INVIERNO', 'PRIMAVERA', 'VERANO', 'OTONO']
Length: 4, dtype: str

• Conteo de días por estación (Total = 366 días):
estacion
VERANO       94
PRIMAVERA    92
INVIERNO     91
OTONO        89
Name: count, dtype: int64


**SOLUCÍON SEMANA_AÑO**
- Se asignaba un 1 a los días 30 y 31 de diciembre de 2024. Sin embargo, no correspnden a la semana 1 del año, si no a la última (53). Asi pues, le asignamos dicho valor a la columna semana_año


In [12]:
# Mostrar todas las filas que tienen semana_año == 1 actualmente
filas_semana_1 = calendario_limpio[calendario_limpio['semana_año'] == 1]

print("===== FILAS DE LA SEMANA_AÑO 1 (TODAS LAS COLUMNAS) =====")
display(filas_semana_1)

===== FILAS DE LA SEMANA_AÑO 1 (TODAS LAS COLUMNAS) =====


,fecha,dia_semana,dia_semana_num,semana_año,mes,trimestre,tipo_dia,festivo,evento,tipo_evento,zona_evento,estacion,es_vispera_festivo,vacaciones_escolares
0,01-01-2024,LUNES,0,1,1,1,festivo,Año Nuevo,NaN,NaN,NaN,invierno,0,1
1,02-01-2024,MARTES,1,1,1,1,laborable,NaN,NaN,NaN,NaN,invierno,0,1
2,03-01-2024,MIERCOLES,2,1,1,1,laborable,NaN,NaN,NaN,NaN,invierno,0,1
3,04-01-2024,JUEVES,3,1,1,1,laborable,NaN,NaN,NaN,NaN,Invierno,0,1
4,05-01-2024,VIERNES,4,1,1,1,puente,NaN,NaN,NaN,NaN,invierno,1,1
5,06-01-2024,SABADO,5,1,1,1,festivo,Reyes Magos,NaN,NaN,NaN,invierno,0,1
6,07-01-2024,DOMINGO,6,1,1,1,fin_semana,NaN,NaN,NaN,NaN,Otoño,0,1
364,30-12-2024,LUNES,0,1,12,4,laborable,NaN,NaN,NaN,NaN,Primavera,0,0
365,31-12-2024,MARTES,1,1,12,4,laborable,NaN,Nochevieja,CELEBRACION,Sol,invierno,0,0


In [14]:
#cambiamos el valor 1 de semana_año en los días 30 y 31 de diciembre a 53 (ya que debemos saber que esos días no pertenecen a la semana 1 del año, si no a los últimos)
#Convertir a datetime para operarlo correctamente
fechas_dt = pd.to_datetime(calendario_limpio['fecha'], format='%d-%m-%Y')

# Recalcular la semana ISO y asignar 53 a los días de finales de diciembre
semanas_iso = fechas_dt.dt.isocalendar().week
calendario_limpio['semana_año'] = [
    53 if (mes == 12 and semana == 1) else semana 
    for mes, semana in zip(fechas_dt.dt.month, semanas_iso)
]

# VERIFICACIÓN
print("===== COMPROBACIÓN TRAS EL CAMBIO =====")
print("• Días en la Semana 1 (debe ser 7):", (calendario_limpio['semana_año'] == 1).sum())
print("• Días en la Semana 53 (debe ser 2):", (calendario_limpio['semana_año'] == 53).sum())

print("\n• Últimas 2 filas del año (30 y 31 de diciembre):")
display(calendario_limpio.tail(2)[['fecha', 'dia_semana', 'semana_año', 'mes', 'trimestre']])

===== COMPROBACIÓN TRAS EL CAMBIO =====
• Días en la Semana 1 (debe ser 7): 7
• Días en la Semana 53 (debe ser 2): 2

• Últimas 2 filas del año (30 y 31 de diciembre):


,fecha,dia_semana,semana_año,mes,trimestre
364,30-12-2024,LUNES,53,12,4
365,31-12-2024,MARTES,53,12,4


**SOLUCIÓN NULOS EN LAS COLUMNAS RESTANTES**
- Al haber mas de un 95% de nulos en las columnas festivo, evento y zona evento, decidimos imputar valores nulos con etiquetas descriptivas explícitas (NO FESTIVO, SIN EVENTO, SIN ZONA) para evitar errores. Asimismo damos consistencia al formato estandarizando textos a mayusculas y sin tildes como hemos hecho en las otras columnas categoricas.

In [21]:
# 1. Rellenar nulos con etiquetas explícitas
calendario_limpio['festivo'] = calendario_limpio['festivo'].fillna('NO_FESTIVO')
calendario_limpio['evento'] = calendario_limpio['evento'].fillna('SIN_EVENTO')
calendario_limpio['zona_evento'] = calendario_limpio['zona_evento'].fillna('SIN_ZONA')

# 2. Convertir a mayúsculas y eliminar tildes/acentos
def normalizar_texto(val):
    if pd.isna(val) or val is None:
        return val
    # Mapeo de tildes
    trans = str.maketrans('ÁÉÍÓÚÁÉÍÓÚaéíóú', 'AEIOUAEIOUAEIOU')
    return str(val).translate(trans).upper().strip()

for col in ['festivo', 'evento', 'zona_evento']:
    calendario_limpio[col] = calendario_limpio[col].apply(normalizar_texto)

# VERIFICACIÓN
print("===== VERIFICACIÓN DE COLUMNAS CON ALTOS NULOS =====")
print("• Nulos en 'festivo':", calendario_limpio['festivo'].isna().sum())
print("• Nulos en 'evento':", calendario_limpio['evento'].isna().sum())
print("• Nulos en 'zona_evento':", calendario_limpio['zona_evento'].isna().sum())

print("\n• Valores únicos en 'zona_evento':")
print(calendario_limpio['zona_evento'].unique())

print("\n• Muestra de días con eventos y festivos:")
display(calendario_limpio[calendario_limpio['evento'] != 'SIN_EVENTO'][['fecha', 'festivo', 'evento', 'tipo_evento', 'zona_evento']].head())

===== VERIFICACIÓN DE COLUMNAS CON ALTOS NULOS =====
• Nulos en 'festivo': 0
• Nulos en 'evento': 0
• Nulos en 'zona_evento': 0

• Valores únicos en 'zona_evento':
<StringArray>
[     'SIN_ZONA',      'BERNABEU',        'CENTRO',       'PRADERA',
        'CHUECA',      'MAD COOL', 'METROPOLITANO',        'CIUDAD',
           'SOL']
Length: 9, dtype: str

• Muestra de días con eventos y festivos:


,fecha,festivo,evento,tipo_evento,zona_evento
13,14-01-2024,NO_FESTIVO,PARTIDO REAL MADRID VS VILLARREAL,DEPORTIVO,BERNABEU
44,14-02-2024,NO_FESTIVO,SAN VALENTIN,CELEBRACION,SIN_ZONA
69,10-03-2024,NO_FESTIVO,PARTIDO REAL MADRID VS BETIS,DEPORTIVO,BERNABEU
111,21-04-2024,NO_FESTIVO,MARATON DE MADRID,DEPORTIVO,CENTRO
135,15-05-2024,SAN ISIDRO,FIESTAS SAN ISIDRO,FIESTAS,PRADERA


In [31]:
# Guardar dataset final completamente limpio y estandarizado
calendario_limpio.to_csv('calendario_limpio.csv', index=False, encoding='utf-8')

print("guardado")

guardado
